#Design Axis: End-to-end vs Decomposed

**Hypothesis:**
Decomposed pipelines will have higher precision because each step is a simpler task.

**Model used:** `google/flan-t5-base` — free, runs on Colab GPU, no API key needed.


In [1]:
# Downloading dataset
import os
from pathlib import Path

BASE      = Path('/content/ebm_nlp_2_00')
DOCS_DIR  = BASE / 'documents'
ANNOT_DIR = BASE / 'annotations' / 'aggregated' / 'hierarchical_labels'

if not BASE.exists():
    print('Downloading EBM-NLP dataset (~30MB)...')
    os.system('wget -q "https://github.com/bepnye/EBM-NLP/raw/master/ebm_nlp_2_00.tar.gz" -O /content/ebm_nlp_2_00.tar.gz')
    os.system('tar -xzf /content/ebm_nlp_2_00.tar.gz -C /content/')
    print('Download complete.')
else:
    print('Dataset already exists.')

assert DOCS_DIR.exists(),  'Documents folder missing'
assert ANNOT_DIR.exists(), 'Annotations folder missing'
print(f'Ready. Documents: {DOCS_DIR}')

Download complete.
Ready. Documents: /content/ebm_nlp_2_00/documents


In [ ]:
# Installing required libraries
!pip install transformers sentencepiece -q

# Loading a QA model instead of Flan-T5
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1
print(f'Using GPU: {device == 0}')

# This model is trained specifically to extract spans from text
# It answers questions like "who are the patients?" by finding the relevant phrase directly in the abstract
qa_pipeline = pipeline(
    'question-answering',
    model='deepset/roberta-base-squad2',
    device=device
)
print('Model ready.')

Using GPU: True


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

In [ ]:
#Testing the model
test_context = 'Forty-five patients with chronic obstructive pulmonary disease received pulmonary rehabilitation before surgery.'
test_question = 'Who are the patients?'
result = qa_pipeline(question=test_question, context=test_context)
print('Test answer:', result['answer'])
print('Confidence:', round(result['score'], 3))


In [ ]:
#Loading a small set of abstracts for extraction.
# We will use 20 abstracts which are enough to show meaningful differences between approaches without taking too long to run the cells.

from pathlib import Path
BASE      = Path('/content/ebm_nlp_2_00')
DOCS_DIR  = BASE / 'documents'
ANNOT_DIR = BASE / 'annotations' / 'aggregated' / 'hierarchical_labels'

def get_all_pmids(split='train'):
    ann_folder = ANNOT_DIR / 'participants' / split
    pmids = []
    for f in ann_folder.iterdir():
        if f.suffix == '.ann':
            pmid = f.stem.split('.')[0]
            if (DOCS_DIR / f'{pmid}.tokens').exists():
                pmids.append(pmid)
    return sorted(pmids)

def load_one_abstract(pmid):
    token_file = DOCS_DIR / f'{pmid}.tokens'
    sent_file  = DOCS_DIR / f'{pmid}.sentences'
    if not token_file.exists():
        return None
    tokens = token_file.read_text(encoding='utf-8', errors='replace').strip().split('\n')
    sentences = []
    if sent_file.exists():
        for line in sent_file.read_text().strip().split('\n'):
            parts = line.strip().split()
            if len(parts) == 2:
                sentences.append((int(parts[0]), int(parts[1])))
    else:
        sentences = [(0, len(tokens))]
    return {'pmid': pmid, 'tokens': tokens, 'sentences': sentences}

def load_ann_labels(pmid, field, split='train'):
    ann_file = ANNOT_DIR / field / split / f'{pmid}.AGGREGATED.ann'
    if not ann_file.exists():
        return []
    labels = []
    for line in ann_file.read_text().strip().split('\n'):
        line = line.strip()
        if line:
            try:
                labels.append(int(line))
            except ValueError:
                labels.append(0)
    return labels

def get_abstract_text(pmid):
    """Return the full abstract as a single string."""
    rec = load_one_abstract(pmid)
    if rec is None:
        return ''
    return ' '.join(rec['tokens'])

def get_gold_spans(pmid, field, split='train'):
    """
    Return the gold-standard extracted text for one field.
    Finds contiguous spans where label != 0 and joins the tokens.
    """
    rec = load_one_abstract(pmid)
    if rec is None:
        return []
    token_labels = load_ann_labels(pmid, field, split)
    if not token_labels:
        return []
    tokens = rec['tokens']
    spans  = []
    current_span = []
    for i, (tok, lab) in enumerate(zip(tokens, token_labels)):
        if lab != 0:
            current_span.append(tok)
        else:
            if current_span:
                spans.append(' '.join(current_span))
                current_span = []
    if current_span:
        spans.append(' '.join(current_span))
    return spans

# Loading 20 abstracts
N = 20
pmids = get_all_pmids()[:N]
print(f'Loaded {len(pmids)} abstracts for extraction')

#example to confirm that gold spans work
ex_pmid = pmids[0]
print(f'\nExample — {ex_pmid}')
for field in ['participants', 'interventions', 'outcomes']:
    spans = get_gold_spans(ex_pmid, field)
    print(f'  Gold {field}: {spans[:2]}')

In [ ]:
# Comparator extraction
# EBM-NLP dataset does not separately label comparators and they are folded
# into the Interventions field. We handle this by detecting comparison
# language within extracted intervention spans and splitting them into
# intervention + comparator components.

COMPARATOR_KEYWORDS = [
    'versus', 'vs', 'compared to', 'compared with',
    'placebo', 'control', 'standard care', 'or'
]

def extract_comparator(intervention_text):
    """
    Splits an intervention span into (intervention, comparator)
    by detecting comparison keywords.
    If no keyword found, returns the full span as intervention
    and empty string as comparator.
    """
    if not intervention_text:
        return intervention_text, ''

    text_lower = intervention_text.lower()

    for keyword in COMPARATOR_KEYWORDS:
        if keyword in text_lower:
            idx          = text_lower.index(keyword)
            intervention = intervention_text[:idx].strip().strip('()')
            comparator   = intervention_text[idx + len(keyword):].strip().strip('()')
            if intervention and comparator:
                return intervention, comparator

    # No comparison keyword found — full span is the intervention
    return intervention_text, ''


print('Comparator extractor ready.')
print()
print('Example keywords:', COMPARATOR_KEYWORDS)

## Approach B — Decomposed 2-step extraction

**Step 1:** Classify each sentence as Participants / Interventions / Outcomes / Other.

**Step 2:** Now for sentences which are classified as relevant, we ask the model to extract the specific span for it.

**Expected:** Higher precision than end-to-end because each step is simpler and more focused.

In [ ]:
#Approach B — Decomposed 2-step
QUESTIONS = {
    'participants': 'Who are the patients or participants in this trial?',
    'interventions': 'What treatment or intervention was given?',
    'outcomes':      'What outcomes or results were measured?'
}

def classify_sentence(sentence):
    """
    Classify a sentence by asking which PICO question it best answers.
    We run all three questions and pick whichever gets the highest confidence.
    """
    best_field = 'other'
    best_score = 0.1

    for field, question in QUESTIONS.items():
        try:
            ans = qa_pipeline(
                question=question,
                context=sentence,
                max_answer_len=50
            )
            if ans['score'] > best_score:
                best_score = ans['score']
                best_field = field
        except Exception:
            continue

    return best_field


def extract_span_from_sentence(sentence, field):
    """Ask the specific question for this field against just this sentence."""
    try:
        ans = qa_pipeline(
            question=QUESTIONS[field],
            context=sentence,
            max_answer_len=80
        )
        return ans['answer'] if ans['score'] > 0.05 else ''
    except Exception:
        return ''


def extract_decomposed_2step(pmid):
    """
    Full 2-step pipeline for one abstract.
    Step 1: classify each sentence
    Step 2: extract span from relevant sentences
    """
    rec = load_one_abstract(pmid)
    if rec is None:
        return {'participants': '', 'interventions': '', 'outcomes': ''}

    tokens    = rec['tokens']
    sentences = rec['sentences']
    result    = {'participants': [], 'interventions': [], 'outcomes': []}

    for start, end in sentences:
        sentence = ' '.join(tokens[start:end]).strip()
        if not sentence:
            continue

        # Step 1: classifying this sentence
        field = classify_sentence(sentence)
        if field == 'other':
            continue

        # Step 2: extracting the specific span
        span = extract_span_from_sentence(sentence, field)
        if span:
            result[field].append(span)

    return {k: ' | '.join(v) for k, v in result.items()}


# Running it on all the 20 abstracts
print('Starting Approach B (decomposed 2-step)...')
print('This takes ~10 minutes — prints each abstract as it goes\n')

results_b = {}
for i, pmid in enumerate(pmids):
    print(f'  [{i+1}/{len(pmids)}] {pmid}', end=' ... ')
    results_b[pmid] = extract_decomposed_2step(pmid)
    print('done')

print('\nAll done!')
print('\nExample output for', pmids[0])
for field, val in results_b[pmids[0]].items():
    print(f'  {field}: {val[:100]}')

## Approach C — Decomposed 3-step extraction

**Approach Steps**: Steps are same as Approach B but we add a cleaning step that:
- Removes duplicate spans
- Removes spans which are too short (noise).
- Removes spans that are clearly wrong field.

**Expected:** Highest precision, but may lose some valid spans (lower coverage).

In [ ]:
# Approach C — Decomposed 3-step (adds cleaning on top of approch B)
def clean_spans(spans_text, field):
    if not spans_text:
        return ''
    spans = [s.strip() for s in spans_text.split('|')]
    cleaned = []
    seen = set()
    for span in spans:
        span = span.strip()
        if len(span.split()) < 2:
            continue
        key = span.lower()
        if key in seen:
            continue
        seen.add(key)
        if field == 'participants':
            person_words = ['patient', 'subject', 'participant', 'adult',
                           'child', 'women', 'men', 'age', 'year',
                           'male', 'female', 'children', 'infant']
            if not any(w in span.lower() for w in person_words):
                continue
        cleaned.append(span)
    return ' | '.join(cleaned)


def extract_decomposed_3step(pmid):
    raw = results_b[pmid]   #Reusing Approach B results and cleaning them.
    return {
        field: clean_spans(raw[field], field)
        for field in ['participants', 'interventions', 'outcomes']
    }


print('Running Approach C (3-step cleaning)...')
results_c = {}
for i, pmid in enumerate(pmids):
    print(f'  [{i+1}/{len(pmids)}] {pmid}', end=' ... ')
    results_c[pmid] = extract_decomposed_3step(pmid)
    print('done')

print('\nAll done!')
print('\nComparison for', pmids[0])
print('  B (raw):    ', results_b[pmids[0]]['participants'])
print('  C (cleaned):', results_c[pmids[0]]['participants'])

In [ ]:
# Side-by-side comparison of all three approaches

print('=' * 80)
print('SIDE-BY-SIDE COMPARISON (first 3 abstracts)')
print('=' * 80)

for pmid in pmids[:3]:
    print(f'\nAbstract: {pmid}')
    print(f'Full text: {get_abstract_text(pmid)[:150]}...')
    print()

    for field in ['participants', 'interventions', 'outcomes']:
        gold  = get_gold_spans(pmid, field)
        gold_str = ' | '.join(gold[:2]) if gold else '(none)'

        b_str = results_b[pmid][field] or '(none)'
        c_str = results_c[pmid][field] or '(none)'

        print(f'  [{field.upper()}]')
        print(f'    Gold : {gold_str[:90]}')

        print(f'    B    : {b_str[:90]}')
        print(f'    C    : {c_str[:90]}')
    print('-' * 80)
    #This output gives us our qualitative findings
    if field == 'interventions':
        i, c = extract_comparator(results_b[pmid]['interventions'])
        print(f'    B intervention : {i[:80]}')
        print(f'    B comparator   : {c[:80] if c else "(none detected)"}')

In [ ]:
#Scoring helper functions
import numpy as np
from pathlib import Path

def load_ann_labels(pmid, field, split='train'):
    ann_file = ANNOT_DIR / field / split / f'{pmid}.AGGREGATED.ann'
    if not ann_file.exists():
        return []
    labels = []
    for line in ann_file.read_text().strip().split('\n'):
        line = line.strip()

        if line:
            try:
                labels.append(int(line))
            except ValueError:
                labels.append(0)
    return labels

def get_gold_tokens(pmid, field):
    rec = load_one_abstract(pmid)
    if rec is None:
        return set()
    token_labels = load_ann_labels(pmid, field)
    if not token_labels:
        return set()
    gold_tokens = set()
    for tok, lab in zip(rec['tokens'], token_labels):
        if lab != 0:
            gold_tokens.add(tok.lower())
    return gold_tokens

def get_pred_tokens(extraction_text):
    if not extraction_text:
        return set()
    tokens = set()
    for part in extraction_text.split('|'):
        for tok in part.strip().split():
            tokens.add(tok.lower().strip('.,;:()[]'))
    return tokens

def token_f1(gold_tokens, pred_tokens):
    if not pred_tokens and not gold_tokens:
        return 1.0, 1.0, 1.0
    if not pred_tokens:
        return 0.0, 0.0, 0.0
    if not gold_tokens:
        return 0.0, 1.0, 0.0
    overlap   = gold_tokens & pred_tokens
    precision = len(overlap) / len(pred_tokens)
    recall    = len(overlap) / len(gold_tokens)
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else 0.0)
    return precision, recall, f1

print('Scoring functions ready.')

In [ ]:
#Calculating F1 scores
FIELDS     = ['participants', 'interventions', 'outcomes']
APPROACHES = {
    'B (2-step)':     results_b,
    'C (3-step)':     results_c,
}

scores = {name: {field: [] for field in FIELDS} for name in APPROACHES}

for name, results in APPROACHES.items():
    for pmid in pmids:
        for field in FIELDS:
            gold_toks = get_gold_tokens(pmid, field)
            pred_toks = get_pred_tokens(results[pmid][field])
            p, r, f1  = token_f1(gold_toks, pred_toks)
            scores[name][field].append((p, r, f1))

print(f"{'Approach':<20} {'Field':<15} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print('-' * 65)
for name in APPROACHES:
    for field in FIELDS:
        vals   = scores[name][field]
        avg_p  = np.mean([v[0] for v in vals])
        avg_r  = np.mean([v[1] for v in vals])
        avg_f1 = np.mean([v[2] for v in vals])
        print(f'{name:<20} {field:<15} {avg_p:>10.3f} {avg_r:>8.3f} {avg_f1:>8.3f}')
    print()

In [ ]:
#Coverage
print(f"{'Approach':<20} {'Field':<15} {'Coverage':>10} {'(out of 20)':>12}")
print('-' * 60)
for name, results in APPROACHES.items():
    for field in FIELDS:
        covered = sum(1 for pmid in pmids if results[pmid][field].strip())
        pct = 100 * covered / len(pmids)
        print(f'{name:<20} {field:<15} {pct:>9.0f}% {covered:>10}/20')
    print()

In [ ]:
#F1 bar chart
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Token-level F1 by approach and field', fontsize=13)

approach_names = list(APPROACHES.keys())
colors = ['#4e8fc7', '#e07b54', '#5aaa7a']
x = np.arange(len(approach_names))

for ax, field in zip(axes, FIELDS):
    f1_vals = [np.mean([v[2] for v in scores[name][field]])
               for name in approach_names]
    bars = ax.bar(x, f1_vals, color=colors, width=0.5)
    ax.set_title(field.capitalize())
    ax.set_xticks(x)
    ax.set_xticklabels(approach_names, fontsize=9)
    ax.set_ylabel('F1 score')
    ax.set_ylim(0, 1)
    for bar, val in zip(bars, f1_vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('f1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved f1_comparison.png')

In [ ]:
# Finding downstream usability
print('=' * 70)
print('DOWNSTREAM USABILITY TEST')
print('Query: Find trials where intervention involves rehabilitation')
print('=' * 70)

for name, results in APPROACHES.items():
    matches = [(pmid, results[pmid]['interventions'])
               for pmid in pmids
               if 'rehabilitation' in results[pmid]['interventions'].lower()]
    print(f'\n{name}: {len(matches)} match(es)')
    for pmid, val in matches:
        print(f'  {pmid}: {val[:100]}')

print()
print('=' * 70)
print('Query 2: Find trials mentioning patients in participants')
print('=' * 70)
for name, results in APPROACHES.items():
    matches = sum(1 for pmid in pmids
                  if 'patient' in results[pmid]['participants'].lower())
    print(f'{name}: {matches} match(es)')

In [ ]:
# Final summary table
print('=' * 75)
print('FULL EVALUATION SUMMARY')
print('=' * 75)
print(f"{'Approach':<20} {'Field':<15} {'P':>6} {'R':>6} {'F1':>6} {'Cover':>7}")
print('-' * 75)

for name, results in APPROACHES.items():
    for field in FIELDS:
        vals     = scores[name][field]
        avg_p    = np.mean([v[0] for v in vals])
        avg_r    = np.mean([v[1] for v in vals])
        avg_f1   = np.mean([v[2] for v in vals])
        coverage = sum(1 for pmid in pmids
                       if results[pmid][field].strip()) / len(pmids)
        print(f'{name:<20} {field:<15} {avg_p:>6.3f} {avg_r:>6.3f} {avg_f1:>6.3f} {coverage:>6.0%}')
    print()

print('P=Precision  R=Recall  F1=F1-score  Cover=fraction with any extraction')

In [ ]:
# Full PICO table
print('=' * 70)
print('FULL PICO TABLE')
print('=' * 70)

for pmid in pmids[:5]:
    print(f'\nAbstract: {pmid}')
    p = results_b[pmid]['participants']
    i_full = results_b[pmid]['interventions']
    o = results_b[pmid]['outcomes']
    i, c = extract_comparator(i_full)

    print(f'  P (Participants) : {p or "(none)"}')
    print(f'  I (Intervention) : {i or "(none)"}')
    print(f'  C (Comparator)   : {c or "(none detected)"}')
    print(f'  O (Outcomes)     : {o or "(none)"}')

In [ ]:
import json
from google.colab import drive, files

# Force remount
drive.mount('/content/drive', force_remount=True)

# Now open the file
drive_path = '/content/drive/MyDrive/Colab Notebooks/05_decomposed_qa_roberta.ipynb'

with open(drive_path) as f:
    nb = json.load(f)

# Fix the metadata
if 'widgets' in nb.get('metadata', {}):
    del nb['metadata']['widgets']

# Save and download
fixed_path = '/content/05_decomposed_qa_roberta_fixed.ipynb'
with open(fixed_path, 'w') as f:
    json.dump(nb, f, indent=1)

print('Fixed. Downloading...')
files.download(fixed_path)